# Chapitre 11 - Exercices 11.1 à 11.4

Ces quatre exercices sont théoriques : lien exact entre réseaux de neurones et projection
pursuit regression (11.1), comportement presque linéaire pour de petits poids (11.2),
rétropropagation pour l'entropie croisée (11.3), et équivalence entre un réseau sans couche
cachée et la régression logistique multinomiale (11.4). Chaque résultat est vérifié
numériquement.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch11_reseaux import (MLP, backprop, forward, gradient_check, init_params,
                              numerical_gradient, one_hot, pack, sigmoid, softmax, unpack)

## Exercice 11.1 : Réseau de neurones et PPR

**Énoncé (reformulé).** Établir la correspondance exacte entre le modèle de projection pursuit
regression (11.1) et le réseau de neurones (11.5). Montrer en particulier qu'un réseau de
régression à une couche cachée équivaut à une PPR dont les fonctions ridge sont
$g_m(\omega_m^T x) = \beta_m\,\sigma(\alpha_{0m} + s_m\,\omega_m^T x)$, où $\omega_m$ est le vecteur
unitaire associé à la $m$-ième unité. Établir une équivalence analogue pour un réseau de
classification.

### Rappels

* **PPR** (éq. (11.1)) : $f(X) = \sum_{m=1}^M g_m(\omega_m^T X)$, avec $\|\omega_m\| = 1$ et des
  fonctions $g_m : \mathbb R\to\mathbb R$ quelconques (estimées par lissage).
* **Réseau** (éq. (11.5)) en régression ($K = 1$, sortie identité) :
  $f(X) = \beta_0 + \sum_{m=1}^M \beta_m\, \sigma(\alpha_{0m} + \alpha_m^T X)$.
* En classification, $T_k = \beta_{0k} + \sum_m \beta_{km} Z_m$ et $f_k(X) = e^{T_k}/\sum_\ell e^{T_\ell}$
  (softmax, éq. (11.6)).

### Solution : cas de la régression

Soit une unité cachée $m$ telle que $\alpha_m \ne 0$. Posons

$$s_m = \|\alpha_m\| > 0, \qquad \omega_m = \frac{\alpha_m}{\|\alpha_m\|}, \quad\text{de sorte que}\quad \|\omega_m\| = 1 \ \text{ et } \ \alpha_m = s_m\,\omega_m.$$

Alors $\alpha_m^T X = s_m\, \omega_m^T X$, et le terme de l'unité $m$ s'écrit

$$\beta_m\, \sigma(\alpha_{0m} + \alpha_m^T X) = \beta_m\,\sigma\big(\alpha_{0m} + s_m\,(\omega_m^T X)\big) = g_m(\omega_m^T X),
\qquad \text{avec } g_m(v) := \beta_m\,\sigma(\alpha_{0m} + s_m v).$$

C'est bien une fonction ridge de direction $\omega_m$. Il reste deux détails :

* la **constante** $\beta_0$ : on l'incorpore à l'un des termes, par exemple
  $g_1(v) := \beta_0 + \beta_1\sigma(\alpha_{01} + s_1 v)$ (ou l'on ajoute une constante au modèle PPR) ;
* une unité avec $\alpha_m = 0$ est **constante** ($\beta_m \sigma(\alpha_{0m})$) : on l'ajoute aussi à la
  constante ; la direction $\omega_m$ est alors arbitraire.

Ainsi $f(X) = \sum_{m=1}^M g_m(\omega_m^T X)$ : **le réseau est une PPR à $M$ termes** dont chaque
fonction ridge appartient à la famille paramétrique à trois paramètres
$\{v \mapsto \beta\,\sigma(\alpha_0 + s v)\}$. Réciproquement, toute PPR dont les $g_m$ sont de cette
forme (avec $s_m > 0$) est un réseau : il suffit de poser $\alpha_m = s_m\omega_m$.
On a donc l'égalité des deux classes de modèles :

$$\{\text{réseaux à } M \text{ unités}\} = \{\text{PPR à } M \text{ termes avec } g_m(v) = \beta_m\sigma(\alpha_{0m} + s_m v)\} \ (+\ \text{constante}).$$

### Solution : cas de la classification

Avec $K$ sorties, chaque combinaison $T_k$ est, par le même calcul,

$$T_k = \beta_{0k} + \sum_{m=1}^M \beta_{km}\,\sigma(\alpha_{0m} + s_m\,\omega_m^T X) = \beta_{0k} + \sum_{m=1}^M g_{km}(\omega_m^T X),
\qquad g_{km}(v) = \beta_{km}\,\sigma(\alpha_{0m} + s_m v).$$

Chaque $T_k$ est donc une PPR, et les $K$ PPR **partagent les mêmes directions** $\omega_1, \dots, \omega_M$
(et les mêmes $\alpha_{0m}$, $s_m$), seules les « hauteurs » $\beta_{km}$ dépendant de $k$.

Le classifieur est un **modèle logistique multinomial** dont les fonctions de score sont ces
PPR : $f_k(X) = e^{T_k}/\sum_\ell e^{T_\ell}$. De façon équivalente, en prenant la classe $K$ comme
référence, les log-rapports de probabilités sont des PPR :

$$\log\frac{f_k(X)}{f_K(X)} = T_k - T_K = (\beta_{0k} - \beta_{0K}) + \sum_{m=1}^M (\beta_{km} - \beta_{Km})\,\sigma(\alpha_{0m} + s_m\,\omega_m^TX).$$

Le réseau de classification correspond donc à une « PPR logistique multinomiale » à directions
communes ; en particulier, pour $K = 2$, $\mathrm{logit}\,P(Y = 1\mid X)$ est une PPR à $M$ termes.

### Vérification numérique

On tire un réseau au hasard ($p = 4$, $M = 5$, $K = 3$), on calcule les directions $\omega_m$ et les
fonctions $g_{km}$, puis on compare les deux écritures.

In [2]:
rng = np.random.RandomState(0)
p, M, K = 4, 5, 3
params = init_params(p, M, K, rng, scale=1.5)
alpha0, alpha = params["alpha"][:, 0], params["alpha"][:, 1:]
beta0, beta = params["beta"][:, 0], params["beta"][:, 1:]

s = np.linalg.norm(alpha, axis=1)             # s_m = ||alpha_m||
omega = alpha / s[:, None]                     # directions unitaires

def g(k, m, v):
    return beta[k, m] * sigmoid(alpha0[m] + s[m] * v)

X = rng.randn(1000, p)
V = X @ omega.T                                # V[i, m] = omega_m^T x_i
T_ppr = np.column_stack([beta0[k] + sum(g(k, m, V[:, m]) for m in range(M)) for k in range(K)])
T_nn = forward(params, X, output="identity")["F"]
print("normes des omega_m :", np.round(np.linalg.norm(omega, axis=1), 12))
print("écart max entre T (réseau) et T (PPR) :", np.max(np.abs(T_nn - T_ppr)))
print("écart max entre les probabilités softmax :",
      np.max(np.abs(forward(params, X, "softmax")["F"] - softmax(T_ppr))))

normes des omega_m : [1. 1. 1. 1. 1.]
écart max entre T (réseau) et T (PPR) : 4.440892098500626e-16
écart max entre les probabilités softmax : 2.220446049250313e-16
